# Day 06 - 模型大比拼：谁更准？前几天的课程里，我们训练了不同的模型。但你怎么知道哪个模型最好呢？今天我们来学习 **如何科学地评估模型**，以及 **如何公平地比较不同模型**。---## 今天的目标1. 理解训练集/测试集划分的原理2. 学习多种评估指标3. 用交叉验证得到更可靠的结果4. 对比多个模型

## 1. 为什么要划分训练集和测试集？想象你在考试：- 如果你背了所有考试题的答案（训练集），考试考的是同样的题 -> 你考了100分，但这不代表你真的会了- 如果考试出了新题（测试集），你还能做对吗？ -> 这才是真正的能力机器学习也是一样！**训练集** 用来教模型，**测试集** 用来检验模型是否真的学会了。

In [ ]:
# 导入库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.svm import SVCfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_scoreplt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]plt.rcParams["axes.unicode_minus"] = False# 读取数据df = pd.read_csv("data/weather_sample.csv", encoding="utf-8-sig")features = ["湿度", "风速", "气压", "温度"]X = df[features]y = df["是否下雨"]print(f"数据量: {len(df)}, 特征数: {len(features)}")

## 2. 四种评估指标准确率 (Accuracy) 不是唯一的标准！| 指标 | 说明 | 什么时候重要 ||------|------|-------------|| 准确率 (Accuracy) | 预测正确的比例 | 数据平衡时 || 精确率 (Precision) | 预测为正例中真正是正例的比例 | 不想误报 || 召回率 (Recall) | 实际正例中被正确预测的比例 | 不想漏报 || F1 分数 | 精确率和召回率的平衡 | 综合考虑时 |

In [ ]:
# 划分数据集X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)# 训练逻辑回归model = LogisticRegression(random_state=42)model.fit(X_train, y_train)y_pred = model.predict(X_test)# 计算四种指标acc = accuracy_score(y_test, y_pred)prec = precision_score(y_test, y_pred)rec = recall_score(y_test, y_pred)f1 = f1_score(y_test, y_pred)print("逻辑回归评估结果:")print(f"  准确率:  {acc*100:.1f}%")print(f"  精确率: {prec*100:.1f}%")print(f"  召回率: {rec*100:.1f}%")print(f"  F1分数: {f1*100:.1f}%")

## 3. 交叉验证 (Cross-Validation)只用一次划分来评估模型，结果可能不太稳定。**交叉验证** 的做法：把数据分成 K 份，轮流用其中 1 份做测试，其余做训练，重复 K 次取平均。

In [ ]:
# 用交叉验证评估多个模型models = {    "逻辑回归": LogisticRegression(random_state=42),    "决策树": DecisionTreeClassifier(max_depth=4, random_state=42),    "K近邻": KNeighborsClassifier(n_neighbors=5),    "支持向量机": SVC(random_state=42),}print("各模型的 5 折交叉验证结果:")print("="*50)results = {}for name, model in models.items():    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")    results[name] = scores    avg = scores.mean()*100    std = scores.std()*100    print(f"{name:10s}: 平均准确率 = {avg:.1f}%, 标准差 = {std:.1f}%")print("="*50)

In [ ]:
# 可视化对比plt.figure(figsize=(10, 6))model_names = list(results.keys())means = [results[n].mean() for n in model_names]stds = [results[n].std() for n in model_names]bars = plt.bar(model_names, means, yerr=stds, capsize=5,              color=["steelblue", "coral", "mediumseagreen", "mediumpurple"])for bar, mean in zip(bars, means):    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,             f"{mean*100:.1f}%", ha="center", fontsize=12)plt.ylabel("准确率", fontsize=14)plt.title("模型交叉验证对比", fontsize=16)plt.ylim(0.5, 1.0)plt.tight_layout()plt.show()

---## 你学到了什么？| 概念 | 说明 ||------|------|| 训练集/测试集 | 教模型 vs 检验模型 || 准确率 | 预测正确的比例 || 精确率 | 预测正例的可靠性 || 召回率 | 找到所有正例的能力 || F1 分数 | 精确率和召回率的平衡 || 交叉验证 | 更可靠的评估方法 |---## 挑战任务（选做）1. 用 F1 分数而不是准确率来做交叉验证对比2. 增加更多模型（如 RandomForest）到对比中3. 研究一下：为什么有时候准确率高但 F1 分数低？